# 第9周 · 优化、正则化与调试（AI 助手开放）

**本周结束时你能做到**：在 2D 曲面上看懂 SGD、Momentum、Adam 的差别，说出学习率与 batch size 的后果；解释初始化为什么重要（对称性、方差逐层放大/缩小、Xavier/Kaiming）；亲手制造过拟合，再用 Dropout、权重衰减、早停、BatchNorm 对付它；掌握"训练诊断配方"的四道体检（第四道：打开网络，读激活与梯度的诊断图）；区分科学 / 冗余 / 固定超参数，用随机搜索在对数尺度上调冗余超参数，并用多种子的均值±标准差报告结果。

本周数据换成 **FashionMNIST**（Xiao, Rasul & Vollgraf 2017, [arXiv:1708.07747](https://arxiv.org/abs/1708.07747)）：和 MNIST 同样 60,000/10,000 张 28×28 灰度图、10 类，但内容是 T 恤、裤子、鞋、包……它的作者说得直白：MNIST 太简单了。第 8 周 PyTorch 官方教程用的就是它。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import copy
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets
from zero2researcher import data, experiments
from zero2researcher.checks import week09 as ck

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

In [ ]:
def load_dataset(cls, train):
    # 下载并缓存（FashionMNIST 约 30MB，只有第一次要等）；网络抖动时自动重试一次
    for attempt in (1, 2):
        try:
            return cls(root=data.download_root(), train=train, download=True)
        except Exception as e:
            if attempt == 1:
                print("下载失败，重试一次：", e)
            else:
                raise


fm_train = load_dataset(datasets.FashionMNIST, train=True)
fm_test = load_dataset(datasets.FashionMNIST, train=False)
pixels = fm_train.data.float() / 255.0
MEAN, STD = pixels.mean().item(), pixels.std().item()      # 第 8 周的"魔法数字"算法


def to_inputs(images):
    # uint8 图像 (N, 28, 28) → 标准化并展平的 float 张量 (N, 784)：一次算好，训练时不再逐张变换
    return ((images.float() / 255.0 - MEAN) / STD).reshape(len(images), -1)


X_val, y_val = to_inputs(fm_train.data[50000:52000]), fm_train.targets[50000:52000]   # 固定验证集
X_test, y_test = to_inputs(fm_test.data), fm_test.targets                               # 测试集全量
print(f"mean = {MEAN:.4f}, std = {STD:.4f} | 验证 {len(X_val)} 张 | 测试 {len(X_test)} 张")

X_tr, y_tr = to_inputs(fm_train.data[:2000]), fm_train.targets[:2000]   # 本周讲义统一的小训练集：前 2000 张
print("训练", len(X_tr), "张 | 类别:", fm_train.classes)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    # 第 8 周作业 A 原样搬来
    model.train()
    total, n = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * len(xb)
        n += len(xb)
    return total / n


def evaluate(model, X, y, device):
    # 第 8 周 evaluate 的"整块张量"版：同样 eval + no_grad，多返回一个平均损失（早停要看验证 loss）
    model.eval()
    with torch.no_grad():
        logits = model(X.to(device))
        y = y.to(device)
        acc = (logits.argmax(dim=1) == y).float().mean().item()
        loss = nn.functional.cross_entropy(logits, y).item()
    return acc, loss


def make_mlp(hidden=256):
    # 本周的"标准小模型"：784-256-10（输入已展平，不需要 Flatten）
    return nn.Sequential(nn.Linear(784, hidden), nn.ReLU(), nn.Linear(hidden, 10))


criterion = nn.CrossEntropyLoss()


def train_epochs(model, X, y, epochs=3, lr=1e-3, batch_size=64, weight_decay=0.0):
    # 用 Adam 把给定模型训练若干 epoch，返回每个 epoch 的平均训练损失（weight_decay 在 S4 的超参数搜索里用到）
    loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    return [train_one_epoch(model, loader, criterion, optimizer, device) for _ in range(epochs)]


def quick_train(X, y, epochs=3, seed=0):
    # 固定种子 → 新建 784-256-10 → 训练，返回 (模型, 每个 epoch 的损失)
    experiments.set_seed(seed)
    model = make_mlp().to(device)
    return model, train_epochs(model, X, y, epochs)

---
## S1 · 优化器与初始化

本节改编自阿姆斯特丹大学深度学习课程的 [UvA DL Tutorial 4: Optimization and Initialization](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/tutorial4/Optimization_and_Initialization.html)（Phillip Lippe），曲面函数与实验设计照搬原文（Momentum 的学习率写法与原文不同，原因见下）。

### 三种优化器在"峡谷"里怎么走

真实网络的损失曲面有上百万维，画不出来；UvA 的办法是造一个**病态的 2D 曲面**：$w_1$ 方向是又窄又陡的峡谷，$w_2$ 方向是一条平缓的长坡。

$$L(w_1, w_2) = \tanh(w_1)^2 + 0.01\,|w_1| + \mathrm{sigmoid}(w_2)$$

三位选手的更新规则（$g$ 是梯度，$\eta$ 是学习率）：

- **SGD**：$w \leftarrow w - \eta g$。每个方向用同一个步长。
- **SGD + Momentum**：$m \leftarrow \beta m + g,\; w \leftarrow w - \eta m$。把历史梯度攒成"惯性"，来回震荡的分量互相抵消、一致的分量越滚越快。
  这是 PyTorch `SGD(momentum=β)` 的写法。UvA 原文写的是滑动平均 $m \leftarrow \beta m + (1-\beta)\,g$，配 lr = 10；两种写法的 $m$ 处处只差一个常数倍 $1/(1-\beta) = 10$，所以换成 PyTorch 写法时等价的学习率是 $10 \times (1 - 0.9) = 1$。下面用 `SGD(lr=1, momentum=0.9)`，轨迹与原文一致；照抄 lr = 10 会直接飞出曲面。
- **Adam**：在 Momentum 之上，再给**每个参数**按其梯度的历史大小单独缩放步长——陡的方向自动迈小步，平的方向自动迈大步。完整的更新式（与 UvA 原文相同；$t$ 是第几步，平方、开方、除法都逐元素，PyTorch 默认 $\beta_1 = 0.9,\ \beta_2 = 0.999,\ \epsilon = 10^{-8}$）：

$$m \leftarrow \beta_1 m + (1-\beta_1)\, g, \qquad v \leftarrow \beta_2 v + (1-\beta_2)\, g^2$$

$$\hat m = \frac{m}{1-\beta_1^{\,t}}, \qquad \hat v = \frac{v}{1-\beta_2^{\,t}}, \qquad w \leftarrow w - \eta\, \frac{\hat m}{\sqrt{\hat v} + \epsilon}$$

  $m$ 是梯度的滑动平均（往哪走），$v$ 是梯度平方的滑动平均（这个方向有多陡）。$\hat m$、$\hat v$ 是**偏差修正**：$m$、$v$ 都从 0 起步，前几步会被严重低估（第 1 步的 $m$ 只有 $0.1\,g$）；除以 $1-\beta^t$ 把它们校正回来，$t$ 一大这个因子就趋近 1，修正自动消失。

让三者从同一点 $(5, 5)$ 出发各走 100 步。下面的循环就是五步核心，只是"模型"换成了两个数：

In [ ]:
def pathological_curve_loss(w1, w2):
    # 照搬 UvA Tutorial 4：w1 方向是陡峭的峡谷，w2 方向是平缓的长坡
    x1_loss = torch.tanh(w1) ** 2 + 0.01 * torch.abs(w1)
    x2_loss = torch.sigmoid(w2)
    return x1_loss + x2_loss


def train_curve(make_optimizer, num_updates=100, init=(5.0, 5.0)):
    # 在 2D 曲面上用给定优化器走 num_updates 步，返回轨迹 (num_updates + 1, 2)
    w = nn.Parameter(torch.tensor(init))
    optimizer = make_optimizer([w])
    path = [w.detach().clone()]
    for _ in range(num_updates):
        # 在这里写你的代码
        raise NotImplementedError
        path.append(w.detach().clone())
    return torch.stack(path).numpy()


paths = {
    "SGD (lr=10)": train_curve(lambda p: torch.optim.SGD(p, lr=10)),
    "SGD + momentum (lr=1, beta=0.9)": train_curve(lambda p: torch.optim.SGD(p, lr=1, momentum=0.9)),
    "Adam (lr=1)": train_curve(lambda p: torch.optim.Adam(p, lr=1)),
}
for name, p in paths.items():
    final = pathological_curve_loss(torch.tensor(p[-1, 0]), torch.tensor(p[-1, 1])).item()
    print(f"{name:34s} 终点 ({p[-1, 0]:6.2f}, {p[-1, 1]:6.2f})   最后 10 步 |w1| 最大 {np.abs(p[-10:, 0]).max():.2f}   终点损失 {final:.3f}")

In [ ]:
allp = np.concatenate(list(paths.values()))
w1g = np.linspace(min(-6, allp[:, 0].min() - 1), max(6, allp[:, 0].max() + 1), 200)
w2g = np.linspace(allp[:, 1].min() - 1, allp[:, 1].max() + 1, 200)
W1, W2 = np.meshgrid(w1g, w2g)
Z = pathological_curve_loss(torch.tensor(W1), torch.tensor(W2)).numpy()

fig, ax = plt.subplots(figsize=(7, 5))
cs = ax.contourf(W1, W2, Z, levels=30, cmap="viridis")
fig.colorbar(cs, label="Loss")
for (name, p), color in zip(paths.items(), ["red", "white", "orange"]):
    ax.plot(p[:, 0], p[:, 1], "o-", ms=2, lw=1, color=color, label=name)
ax.scatter([5], [5], color="black", zorder=5, label="Start")
ax.set_xlabel("w1 (steep valley direction)")
ax.set_ylabel("w2 (gentle slope direction)")
ax.set_title("SGD bounces across the steep valley; momentum and Adam settle in it")
ax.legend(loc="lower left", fontsize=8)
plt.show()

SGD 在陡峭的 $w_1$ 方向上来回横跳、始终落不到谷底（学习率对这个方向太大了），可是换小学习率，$w_2$ 方向的长坡又走不动——**一个全局步长没法同时照顾陡和平两个方向**。Momentum 让来回的分量互相抵消；Adam 给每个方向各配一个步长。这就是 Adam 成为"默认起点"的原因（Karpathy 的 Recipe 里写的就是先用 Adam 3e-4）。

### 学习率：太小、太大、刚好

回到真实网络。同一个 784-256-10、同一份 2000 张训练集，只改 Adam 的学习率，记下**每个 iteration** 的损失：

In [ ]:
def loss_per_iteration(lr, epochs=3, seed=0):
    experiments.set_seed(seed)
    model = make_mlp().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)
    losses = []
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = criterion(model(xb), yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
    return np.array(losses)


lr_curves = {lr: loss_per_iteration(lr) for lr in (1e-4, 1e-3, 1e-2, 1e-1)}
for lr, c in lr_curves.items():
    print(f"lr = {lr:g}: 前 10 步平均 {c[:10].mean():.3f} → 最后 10 步平均 {c[-10:].mean():.3f}   (最大值 {c.max():.1f})")

In [ ]:
def smooth(x, k=10):
    return np.convolve(x, np.ones(k) / k, mode="valid")


fig, ax = plt.subplots(figsize=(7, 3.5))
for lr, c in lr_curves.items():
    ax.plot(smooth(c), label=f"lr = {lr:g}")
ax.set_ylim(0, 3)
ax.set_xlabel("Iteration (moving average of 10)")
ax.set_ylabel("Training loss")
ax.set_title("Too small crawls, too large blows up first: 1e-3 to 1e-2 work here")
ax.legend()
plt.show()

### batch size：小 batch 的梯度是带噪声的估计

一个 mini-batch 的梯度只是"全数据梯度"的**抽样估计**。在同一组参数上，对不同大小的 batch 各抽 20 次，量一量它离全数据梯度有多远：

In [ ]:
experiments.set_seed(0)
probe = make_mlp().to(device)


def first_layer_grad(xb, yb):
    probe.zero_grad()
    criterion(probe(xb.to(device)), yb.to(device)).backward()
    return probe[0].weight.grad.flatten().clone()


full_grad = first_layer_grad(X_tr, y_tr)          # 2000 张全部算进去的"真"梯度
gen = torch.Generator().manual_seed(0)
batch_sizes = [8, 32, 128, 512]
rel_noise = []
for B in batch_sizes:
    errs = []
    for _ in range(20):
        idx = torch.randperm(len(X_tr), generator=gen)[:B]
        g = first_layer_grad(X_tr[idx], y_tr[idx])
        errs.append(((g - full_grad).norm() / full_grad.norm()).item())
    rel_noise.append(np.mean(errs))
    print(f"batch size {B:4d}: 与全数据梯度的相对误差 {rel_noise[-1]:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.loglog(batch_sizes, rel_noise, "o-", label="Measured")
ref = rel_noise[0] * np.sqrt(batch_sizes[0] / np.array(batch_sizes))
ax.loglog(batch_sizes, ref, "--", color="gray", label="1/sqrt(B) reference")
ax.set_xlabel("Batch size B")
ax.set_ylabel("Relative gradient error")
ax.set_title("Gradient noise shrinks roughly like 1/sqrt(B)")
ax.legend()
plt.show()

batch 大 4 倍，噪声只降一半（$1/\sqrt{B}$）——而每个 epoch 的更新步数却少了 4 倍。小 batch：步多、噪声大（噪声有时还有助于跳出尖锐的极小值）；大 batch：步少、每步准、GPU 利用率高。实践中的规则（来自本周 S4 要讲的 Tuning Playbook）：**batch size 主要决定训练速度，不要拿它去调验证集表现**——选硬件放得下的、训练够快的那个，然后重新调学习率。

### 初始化：一切从哪里开始

**① 常数初始化 → 对称性**。如果同一层所有神经元的初始权重一样，它们算出同样的输出、拿到同样的梯度、做同样的更新——**永远是同一个神经元的复制品**。全 0 是最极端的情况：ReLU 网络里除了输出层的 bias，其余参数的梯度全是 0（隐层输出全是 0，输出层权重也是 0，梯度传不过去）——网络只能学到"哪个类最常见"。

In [ ]:
def constant_init(model, value):
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.constant_(m.weight, value)
            nn.init.zeros_(m.bias)


for value in (0.0, 0.005, None):
    experiments.set_seed(0)
    net = make_mlp().to(device)
    if value is not None:
        constant_init(net, value)
    train_epochs(net, X_tr, y_tr, epochs=1)
    W = net[0].weight
    spread = (W.max(dim=0).values - W.min(dim=0).values).max().item()   # 256 个神经元的权重彼此差多少
    acc = evaluate(net, X_test, y_test, device)[0]
    label = "PyTorch default" if value is None else f"constant {value}"
    print(f"{label:16s}: 第一层 256 个神经元之间的最大权重差 {spread:.2e} | 测试准确率 {acc:.3f}")

常数初始化训练一个 epoch 之后，256 个神经元的权重差**恰好为 0**（全 0 时隐层权重的梯度为 0，权重根本没动）——256 个隐层单元等于 1 个。**随机初始化的第一个作用是打破对称**。

**② 方差太小或太大 → 信号逐层消失或爆炸**。每过一层线性层，输出的方差约是输入方差乘以 $d_{\text{in}} \cdot \mathrm{Var}(W)$。想让信号穿过很多层还保持原样，就要让这个乘数 ≈ 1：

- **Xavier/Glorot**（线性或 tanh 网络）：$\mathrm{Var}(W) = 1/d_{\text{in}}$（原论文取 $2/(d_{\text{in}} + d_{\text{out}})$，思想相同）；
- **Kaiming/He**（ReLU 网络）：$\mathrm{Var}(W) = 2/d_{\text{in}}$——ReLU 把一半的输入砍成 0，方差减半，所以要补一个 2。

在一个 8 层、每层 256 宽的 ReLU 网络里，用四种初始化各喂进 512 张图，看每一层线性输出的分布：

In [ ]:
def make_deep(depth=8, width=256):
    layers, d = [], 784
    for _ in range(depth):
        layers += [nn.Linear(d, width), nn.ReLU()]
        d = width
    layers.append(nn.Linear(d, 10))
    return nn.Sequential(*layers)


def init_normal(model, std_fn):
    # std_fn(d_in) 给出这一层权重的标准差
    for m in model.modules():
        if isinstance(m, nn.Linear):
            # 在这里写你的代码
            raise NotImplementedError


def hidden_preactivations(model, x):
    # 逐层记录每个隐层 Linear 的输出（激活函数之前）
    outs, h = [], x
    with torch.no_grad():
        for layer in model:
            h = layer(h)
            if isinstance(layer, nn.Linear):
                outs.append(h.flatten().cpu())
    return outs[:-1]


inits = {
    "std = 0.01 (too small)": lambda d: 0.01,
    "std = 0.2 (too large)": lambda d: 0.2,
    "Xavier: sqrt(1/d_in)": lambda d: (1 / d) ** 0.5,
    "Kaiming: sqrt(2/d_in)": lambda d: (2 / d) ** 0.5,
}
x_batch = X_tr[:512].to(device)
acts = {}
for name, fn in inits.items():
    experiments.set_seed(0)
    deep = make_deep().to(device)
    init_normal(deep, fn)
    acts[name] = hidden_preactivations(deep, x_batch)
    print(f"{name:24s} 各层标准差: " + " ".join(f"{a.std().item():.2g}" for a in acts[name]))

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(10, 7))
rows = ["std = 0.01 (too small)", "std = 0.2 (too large)", "Kaiming: sqrt(2/d_in)"]
for r, name in enumerate(rows):
    for c, layer in enumerate([0, 3, 7]):
        a = acts[name][layer].numpy()
        axes[r, c].hist(a, bins=60, color=f"C{r}")
        axes[r, c].set_title(f"{name.split(' (')[0]} | layer {layer + 1} | std {a.std():.2g}", fontsize=9)
        axes[r, c].set_xlabel("Pre-activation value")
        axes[r, c].set_ylabel("Count")
fig.suptitle("Bad init shrinks or blows up the signal layer by layer; Kaiming keeps its scale")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 3.5))
for name, a in acts.items():
    ax.semilogy(range(1, 9), [t.std().item() for t in a], "o-", label=name)
ax.set_xlabel("Hidden layer")
ax.set_ylabel("Std of pre-activations (log scale)")
ax.set_title("Only Kaiming keeps a ReLU net's activations stable across 8 layers")
ax.legend(fontsize=8)
plt.show()

注意直方图的**横轴刻度**：std = 0.01 时到第 8 层只剩 $10^{-8}$ 量级（信号与梯度一起消失），std = 0.2 时涨到上千（爆炸），Kaiming 一路保持在同一量级。Xavier 在 ReLU 网络里每层缩小约 $1/\sqrt{2}$——8 层后小了一个数量级：**初始化要和激活函数配套**。

PyTorch 的 `nn.Linear` 默认初始化比 Xavier 还要小一些（方差约 $1/(3 d_{\text{in}})$）。浅网络无所谓；网络一深，信号就会逐层变弱——记住这一点，本周作业里 8 层网络的表现会让你想起它。

---
## S2 · 过拟合与正则化

### 先亲手制造过拟合

配方：**数据少（2000 张）+ 模型大（784-512-256-10，约 53 万参数）+ 训练久（30 epochs）**。每个 epoch 记下训练集与验证集上的损失和准确率：

In [ ]:
def make_net(dropout=0.0, batchnorm=False):
    # 784-512-256-10；可选在每个隐层加 BatchNorm 与 Dropout
    layers, d = [], 784
    for width in (512, 256):
        # 在这里写你的代码
        raise NotImplementedError
    layers.append(nn.Linear(d, 10))
    return nn.Sequential(*layers)


def fit(make_model, epochs=30, lr=1e-3, weight_decay=0.0, patience=None, seed=0):
    # 训练并逐 epoch 记录曲线；patience 不为 None 时按验证 loss 早停，并恢复验证 loss 最低时的权重
    experiments.set_seed(seed)
    model = make_model().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=128, shuffle=True)
    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_loss, best_state, bad = float("inf"), None, 0
    for epoch in range(epochs):
        train_one_epoch(model, loader, criterion, optimizer, device)
        tr_acc, tr_loss = evaluate(model, X_tr, y_tr, device)
        va_acc, va_loss = evaluate(model, X_val, y_val, device)
        for key, v in zip(hist, (tr_loss, va_loss, tr_acc, va_acc)):
            hist[key].append(v)
        if patience is not None:
            # 在这里写你的代码
            raise NotImplementedError
    if best_state is not None:
        model.load_state_dict(best_state)
    hist["final_train_acc"] = evaluate(model, X_tr, y_tr, device)[0]
    hist["test_acc"] = evaluate(model, X_test, y_test, device)[0]
    hist["epochs_run"] = len(hist["train_loss"])
    return model, hist


_, base = fit(make_net)
print(f"训练准确率 {base['final_train_acc']:.3f} | 测试准确率 {base['test_acc']:.3f} | "
      f"gap {base['final_train_acc'] - base['test_acc']:.3f} | "
      f"验证 loss 最低在第 {int(np.argmin(base['val_loss'])) + 1} 个 epoch")
ck.overfit_gap(base["final_train_acc"], base["test_acc"])

In [ ]:
ep = range(1, base["epochs_run"] + 1)
best_ep = int(np.argmin(base["val_loss"])) + 1
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(ep, base["train_loss"], label="Train")
axes[0].plot(ep, base["val_loss"], label="Validation")
axes[0].axvline(best_ep, color="gray", ls="--", label=f"Lowest val loss (epoch {best_ep})")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy loss")
axes[0].set_title("Validation loss bottoms out early, then climbs")
axes[0].legend()
axes[1].plot(ep, base["train_acc"], label="Train")
axes[1].plot(ep, base["val_acc"], label="Validation")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Train accuracy keeps rising while validation stalls")
axes[1].legend()
plt.tight_layout()
plt.show()

左图是过拟合的标准签名：训练损失一路往下，**验证损失先降后升**——从那个拐点开始，模型在背训练集，而不是在学"衣服长什么样"。

### 四件武器

| 方法 | 做什么 | PyTorch |
| --- | --- | --- |
| **Dropout**（Srivastava et al. 2014） | 训练时每个神经元以概率 p 被随机关掉，逼网络不依赖任何单个神经元；评估时全开 | `nn.Dropout(0.5)`——受 `model.train()/eval()` 控制，第 8 周的习惯在这里兑现 |
| **权重衰减** | 损失里加 $\tfrac{\lambda}{2} \lVert w \rVert^2$（等价于每一步在梯度上加 $\lambda w$），惩罚大权重 | `Adam(..., weight_decay=1e-3)`——参数 `weight_decay` 就是这里的 $\lambda$，PyTorch 把它直接加在梯度上 |
| **早停** | 盯着验证 loss，连续 `patience` 个 epoch 不创新低就停，并退回最好的那次 | 手写（上面 `fit` 里的那几行） |
| **BatchNorm**（Ioffe & Szegedy 2015） | 把每层的输出标准化：训练时用当前 batch 的均值/方差（并顺手更新一份 running 均值/方差），评估时改用 running 统计。主要作用是让训练更稳更快，附带一点正则化效果 | `nn.BatchNorm1d(width)`——和 Dropout 一样受 `model.train()/eval()` 控制（表格下方有演示） |

逐个加上，其余一切不变（同一种子、同一数据、同样 30 epochs）：

In [ ]:
configs = {
    "Baseline": dict(make_model=make_net),
    "Dropout 0.5": dict(make_model=lambda: make_net(dropout=0.5)),
    "Weight decay 1e-4": dict(make_model=make_net, weight_decay=1e-4),
    "Weight decay 1e-3": dict(make_model=make_net, weight_decay=1e-3),
    "Weight decay 1e-2": dict(make_model=make_net, weight_decay=1e-2),
    "Early stopping (patience 5)": dict(make_model=make_net, patience=5),
    "BatchNorm": dict(make_model=lambda: make_net(batchnorm=True)),
}
reg = {"Baseline": base}
for name, kw in configs.items():
    if name != "Baseline":
        reg[name] = fit(**kw)[1]
for name, h in reg.items():
    print(f"{name:28s} 训练 {h['final_train_acc']:.3f} | 测试 {h['test_acc']:.3f} | "
          f"gap {h['final_train_acc'] - h['test_acc']:.3f} | 最终验证 loss {h['val_loss'][-1]:.3f} | "
          f"跑了 {h['epochs_run']} 个 epoch")

In [ ]:
names = list(reg)
x = np.arange(len(names))
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.bar(x - 0.2, [reg[n]["final_train_acc"] for n in names], width=0.4, label="Train")
bars = ax.bar(x + 0.2, [reg[n]["test_acc"] for n in names], width=0.4, label="Test")
ax.bar_label(bars, fmt="%.3f", fontsize=8)
ax.set_xticks(x, names, rotation=20, ha="right")
ax.set_ylim(0.7, 1.02)
ax.set_ylabel("Accuracy")
ax.set_title("Regularisers shrink the gap mostly by lowering train accuracy; test barely moves")
ax.legend()
plt.tight_layout()
plt.show()

诚实地读这张表：Dropout、强权重衰减、早停确实把 gap 收窄了 4–8 个百分点，但**主要是把训练准确率拉了下来，测试准确率几乎没动**（BatchNorm 在这个规模上甚至更低）。它们真正改善的是**验证 loss**（看上一格的最后一列：baseline 约 0.76，Dropout 约 0.57，早停约 0.50）——没有正则化的模型对错误的预测也信心十足，少数高置信的错误把损失拉得很高；正则化让模型不那么自负，但分对的图并没有变多。

两个教训：

1. **gap 变小 ≠ 泛化变好**。gap 可以靠"训练集上变差"来缩小；判断正则化有没有用，要看验证/测试指标本身。
2. 2000 张图里能学到的东西就这么多。Karpathy 在 Recipe 里说得最直接：最好的正则化永远是**更多的真实数据**，其次才是这些技巧。

还有一个细节值得盯住：Dropout 那一行的"训练准确率"是在 `model.eval()` 下测的（Dropout 关闭）。如果你在训练模式下测，会得到一个偏低的数——**评估永远在 eval 模式下做**。

BatchNorm 在两种模式下的差别更大，也更隐蔽（第 8 周预告的"下周见真章"）：

- **train 模式**：用**当前这个 batch** 的均值和方差做标准化，同时以滑动平均的方式把它们记进 `running_mean` / `running_var`——第 8 周演示过，哪怕包在 `no_grad` 里也照记不误；
- **eval 模式**：不再看 batch，改用训练时攒下的 running 统计，也不再更新它们。

所以评估时忘了 `model.eval()`，同一张图的预测会**随它和哪些图分在同一个 batch 而变**，running 统计还会被测试数据改写。看一个最小的例子：

In [ ]:
# 同一张图放进两个不同的 batch：train 模式下它的输出跟着 batch 变，eval 模式下不变
experiments.set_seed(0)
bn = nn.BatchNorm1d(3)
img = torch.randn(1, 3)                                    # "同一张图"（3 个特征）
batch_a = torch.cat([img, torch.randn(15, 3)])             # 和 15 张别的图凑成一个 batch
batch_b = torch.cat([img, torch.randn(15, 3) + 2.0])       # 换一批同伴（整体偏亮）
with torch.no_grad():
    bn.train()
    print("train 模式：", bn(batch_a)[0].numpy().round(3), "vs", bn(batch_b)[0].numpy().round(3))
    print("            running_mean 已被这两次前向改成", bn.running_mean.numpy().round(3))
    bn.eval()
    print("eval 模式： ", bn(batch_a)[0].numpy().round(3), "vs", bn(batch_b)[0].numpy().round(3))

---
## S3 · 训练诊断配方

模型训不好的时候，**别急着调超参数**。Karpathy 的 [A Recipe for Training Neural Networks](https://karpathy.github.io/2019/04/25/recipe/)（2019）给出一套先体检、后治疗的流程；今天学其中最便宜、最常救命的三道体检，再加一道"打开网络看内部"。

### 体检 ①：初始损失对不对

随机初始化、还没训练的 10 类分类器应该"什么都不知道"：对每类给出差不多的概率 1/10，于是交叉熵

$$L_{\text{init}} \approx -\ln\frac{1}{10} = \ln 10 \approx 2.303$$

明显偏离这个数（尤其偏低），说明在训练之前就有东西不对：

In [ ]:
experiments.set_seed(0)
model = make_mlp().to(device)
xb, yb = X_tr[:256].to(device), y_tr[:256].to(device)
# 在这里写你的代码
raise NotImplementedError
print(f"初始损失 {init_loss:.4f}   理论值 ln(10) = {np.log(10):.4f}")
ck.init_loss(init_loss)

比 2.303 略高一点——PyTorch 默认初始化下最后一层输出的 logits 不够小，各类概率不完全均等。Recipe 要求"把最后一层初始化对"（原文举的例子是设置末层的 **bias**：比如回归目标的均值是 50，就把末层 bias 初始化为 50）。对类别均衡的分类问题，一个常用的做法是把末层**权重**缩小 100 倍、bias 清零（Karpathy 在 makemore 系列第 3 讲里演示过这一招）：logits 几乎全为 0，损失就应精确地落在 ln 10：

In [ ]:
# 在这里写你的代码
raise NotImplementedError
print(f"末层权重 ×0.01 之后的初始损失 {scaled_loss:.4f}")

### 体检 ②：先过拟合一个极小的 batch

Recipe 原话的意思：拿少至两张图，把模型训到损失接近 0。**连两张图都背不下来的模型/训练循环，一定有 bug**——这道体检能在几秒内排除一大类错误（损失函数接错、梯度没传回去、优化器没拿到参数……）：

In [ ]:
experiments.set_seed(0)
model = make_mlp().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
x2, y2 = X_tr[:2].to(device), y_tr[:2].to(device)
# 在这里写你的代码
raise NotImplementedError
print(f"200 步之后两张图上的损失 {loss.item():.6f}")
ck.overfit_one_batch(loss.item())

### 体检 ③：与输入无关的基线

把所有输入都换成 0 再训练——此时模型只能学到"哪个类最常见"，测试准确率应退化到多数类水平（FashionMNIST 各类均衡，所以是 0.10），损失停在 ln 10 附近。**用真实输入的模型必须明显好过它**；如果两者差不多，说明模型根本没在用输入（数据管道断了、标签和图片对不上……）：

In [ ]:
# 在这里写你的代码
raise NotImplementedError
real_model, real_losses = quick_train(X_tr, y_tr)
real_acc = evaluate(real_model, X_test, y_test, device)[0]
print(f"输入全为 0：测试准确率 {zero_acc:.3f}，损失 {zero_loss:.3f}")
print(f"真实输入：  测试准确率 {real_acc:.3f}   ← 必须明显高于上一行")

### 体检 ④：打开网络看内部（Karpathy makemore 第 3 讲）

前三道体检只看网络的输入和输出。Recipe 也提到要看网络**内部**的激活有没有怪异之处；Karpathy 在 Neural Networks: Zero to Hero 系列 makemore 第 3 讲的结尾（[视频](https://youtu.be/P6sfmUTpUmc)、[notebook](https://github.com/karpathy/nn-zero-to-hero/blob/master/lectures/makemore/makemore_part3_bn.ipynb)，MIT 许可），把它做成了一套固定的诊断图：每一层的激活和梯度长什么样。下面照搬他的网络结构和诊断写法，只把输入从名字里的字符换成本周的 FashionMNIST。

**网络**：5 × [Linear(100) → (BatchNorm) → Tanh]，再接 Linear(100, 10)。本周其他地方都用 ReLU，这里换成 tanh，是因为 tanh 的毛病最直观：它的输出被夹在 (−1, 1) 之间，局部导数是 $1 - t^2$。$|t| > 0.97$ 时导数不到 0.06，梯度几乎过不去——这个神经元**饱和**了。（ReLU 对应的毛病是"死神经元"：输入恒为负，输出恒为 0。）

**初始化**：和 S1 一样，权重从标准差为 $\text{gain}/\sqrt{d_{\text{in}}}$ 的正态分布里抽。tanh 推荐的 gain 是 5/3（`nn.init.calculate_gain('tanh')` 返回的就是它），用来抵消 tanh 对信号的压缩。末层权重再乘 0.1、bias 清零——体检 ① 那一招，让初始预测别太自信。

**四种诊断**，都在一次 SGD 训练（lr = 0.1、batch 32、1000 步）里顺手记录：

| 图 | 看什么 | 健康的样子 |
| --- | --- | --- |
| ① 各 tanh 层的激活直方图 | 饱和比例：$\lvert t \rvert > 0.97$ 的元素占多少 | 第 2 层以后各层相近，饱和比例在百分之几到一成以内（Karpathy 那次带 BN 的运行约 2–3%） |
| ② 各 tanh 层**输出的梯度**直方图 | 梯度往回传时是变大还是变小 | 各层分布相近 |
| ③ 每个权重矩阵的 grad:data | 梯度的标准差 / 参数的标准差 | 各层量级相近 |
| ④ 每个权重矩阵的 update:data（取 log10）随步数的曲线 | 这一步的**更新量**相对参数本身有多大 | 在 $10^{-3}$ 附近（图上画一条 −3 的参照线） |

第④张最常用：SGD 的更新量就是 lr × 梯度，所以 update:data 直接告诉你学习率对这一层是太大还是太小。Karpathy 在原 notebook 里写的经验值是约 1e-3；同一个经验值也写在 CS231n 的笔记里（[Neural Networks Part 3](https://cs231n.github.io/neural-networks-3/)）：这个比"somewhere around 1e-3"，明显更低说明学习率可能太小，明显更高说明可能太大。它只是**量级参照**，不是硬标准。

先搭网络（照搬，给出）：

In [ ]:
# 网络结构与初始化照搬 Karpathy makemore 第 3 讲（karpathy/nn-zero-to-hero，MIT 许可）：
# https://github.com/karpathy/nn-zero-to-hero/blob/master/lectures/makemore/makemore_part3_bn.ipynb
# 改动：输入换成 FashionMNIST 的 784 维像素；用 nn.Sequential 搭，代替原文手写的 Linear / BatchNorm1d / Tanh 类；
# 有 BatchNorm 时末层也统一用"权重 ×0.1"（原文 BN 版在末层多一个 BatchNorm1d，把它的 γ 乘 0.1）
def make_tanh_net(gain=5 / 3, batchnorm=False, depth=5, width=100):
    layers, d = [], 784
    for _ in range(depth):
        layers.append(nn.Linear(d, width, bias=not batchnorm))   # 有 BatchNorm 时 bias 多余：BN 会减掉均值
        if batchnorm:
            layers.append(nn.BatchNorm1d(width))
        layers.append(nn.Tanh())
        d = width
    layers.append(nn.Linear(d, 10))
    net = nn.Sequential(*layers)
    with torch.no_grad():
        for m in net:
            if isinstance(m, nn.Linear) and gain is not None:     # gain=None：保留 PyTorch 的默认初始化
                nn.init.normal_(m.weight, std=gain / m.in_features ** 0.5)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
        net[-1].weight.mul_(0.1)          # 末层缩小，初始预测别太自信（原文：layers[-1].weight *= 0.1）
        net[-1].bias.zero_()
    return net


print(make_tanh_net(batchnorm=True))

**第一步：带探头的前向**。逐层前向，遇到 `nn.Tanh` 就把它的输出存进 `taps`。中间结果（非叶子张量）的 `.grad` 默认会在反向传播之后丢掉，所以存之前要对它调用 `.retain_grad()`——Karpathy 原文里的 `layer.out.retain_grad()`：

In [ ]:
def forward_with_taps(net, x):
    # 逐层前向；每个 Tanh 的输出都存进 taps，并让它在反向传播后保留 .grad
    taps = []
    for layer in net:
        x = layer(x)
        if isinstance(layer, nn.Tanh):
            # 在这里写你的代码
            raise NotImplementedError
    return x, taps


ck.forward_with_taps(forward_with_taps)

**第二步：三个诊断量**，各一行，照搬 Karpathy 的写法：

- 饱和比例：张量 `t` 里 $|t| > 0.97$ 的元素占多少（比例，不是百分数）；
- grad:data：权重梯度的标准差 / 权重本身的标准差；
- update:data：这一步的更新量（SGD 下是 `lr * p.grad`）的标准差 / 权重的标准差，再取 log10。

三个函数都返回 0 维张量即可（训练循环里每一步都要算 update:data，留在张量上、最后一次性取出来，比每步 `.item()` 快）：

In [ ]:
def saturation(t):
    # tanh 输出 t 中饱和（|t| > 0.97）的比例
    # 在这里写你的代码
    raise NotImplementedError


def grad_data_ratio(p):
    # 权重 p 的 grad:data
    # 在这里写你的代码
    raise NotImplementedError


def update_data_ratio(p, lr):
    # SGD 一步的 log10(update:data)
    # 在这里写你的代码
    raise NotImplementedError


ck.diagnostic_stats(saturation, grad_data_ratio, update_data_ratio)

**第三步：边训练边记录**。训练循环照 Karpathy 手写 SGD（`p -= lr * p.grad`），每一步记下各权重矩阵的 update:data；第 0 步（刚初始化）和最后一步各存一份"快照"：激活、激活的梯度、饱和比例、grad:data。三种配置：好的初始化（gain 5/3）、坏的初始化（gain 3）、坏的初始化 + BatchNorm（S2 学过的 BN）。三种配置合计 CPU 上约 3 秒：

In [ ]:
def train_with_diagnostics(net, steps=1000, lr=0.1, batch_size=32, seed=0):
    # 手写 SGD（与 Karpathy 一致），每步记录 update:data；第 0 步与最后一步各存一份快照
    gen = torch.Generator().manual_seed(seed)                      # 抽 batch 的随机数单独固定
    weights = [p for p in net.parameters() if p.ndim == 2]         # 只看二维的权重矩阵（原文同样如此）
    ud, snaps = [], {}
    for step in range(steps):
        idx = torch.randint(0, len(X_tr), (batch_size,), generator=gen)
        logits, taps = forward_with_taps(net, X_tr[idx].to(device))
        loss = criterion(logits, y_tr[idx].to(device))
        net.zero_grad()
        loss.backward()
        with torch.no_grad():
            if step in (0, steps - 1):
                snaps["init" if step == 0 else "end"] = {
                    "acts": [t.detach().cpu() for t in taps],
                    "grads": [t.grad.cpu() for t in taps],
                    "sat": [float(saturation(t)) for t in taps],
                    "grad_data": [float(grad_data_ratio(p)) for p in weights],
                }
            for p in net.parameters():
                p -= lr * p.grad
            ud.append(torch.stack([torch.as_tensor(update_data_ratio(p, lr)) for p in weights]))
    return snaps, torch.stack(ud).cpu().numpy()                    # ud 的形状：(步数, 权重矩阵个数)


tanh_configs = {"gain 5/3": dict(gain=5 / 3), "gain 3": dict(gain=3.0), "BN + gain 3": dict(gain=3.0, batchnorm=True)}
diag = {}
for name, kw in tanh_configs.items():
    experiments.set_seed(0)
    tanh_net = make_tanh_net(**kw).to(device)
    snaps, ud = train_with_diagnostics(tanh_net)
    diag[name] = {"sat_init": snaps["init"]["sat"], "sat_end": snaps["end"]["sat"], "ud": ud, "snaps": snaps}
    pct = lambda v: " ".join(f"{100 * s:4.1f}%" for s in v)
    print(f"{name:12s} 饱和比例：初始 {pct(snaps['init']['sat'])}  →  1000 步后 {pct(snaps['end']['sat'])}")
    print(f"{'':12s} log10(update:data) 最后 100 步：第 1 层 {ud[-100:, 0].mean():.2f}，第 2–5 层 {ud[-100:, 1:-1].mean():.2f}"
          f" | 验证准确率 {evaluate(tanh_net, X_val, y_val, device)[0]:.3f}")
ck.tanh_diagnostics(diag)

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(17, 10))
for r, (name, d) in enumerate(diag.items()):
    s = d["snaps"]["init"]                                   # 前三列：刚初始化时（第 0 步）
    ax = axes[r, 0]
    for i, t in enumerate(s["acts"]):
        hy, hx = torch.histogram(t.flatten(), bins=50, density=True)
        ax.plot(hx[:-1], hy, label=f"tanh {i + 1}: sat {100 * s['sat'][i]:.0f}%")
    ax.set_title(f"{name} | activations (step 0)")
    ax.set_xlabel("tanh output")
    ax.set_ylabel("Density")
    ax.legend(fontsize=7)
    ax = axes[r, 1]
    for i, g in enumerate(s["grads"]):
        hy, hx = torch.histogram(g.flatten(), bins=50, density=True)
        ax.plot(hx[:-1], hy, label=f"tanh {i + 1}: std {g.std():.1e}")
    ax.set_title(f"{name} | gradients of activations (step 0)")
    ax.set_xlabel("Gradient")
    ax.set_ylabel("Density")
    ax.legend(fontsize=7)
    ax = axes[r, 2]
    n_w = len(s["grad_data"])
    ax.bar([f"W{i + 1}" + ("\n(output)" if i == n_w - 1 else "") for i in range(n_w)], s["grad_data"])
    ax.set_yscale("log")
    ax.set_title(f"{name} | grad:data per weight matrix (step 0)")
    ax.set_ylabel("grad std / weight std")
    ax = axes[r, 3]                                          # 第四列：全程
    for i in range(n_w):
        ax.plot(d["ud"][:, i], lw=0.7, label=f"W{i + 1}" + (" (output)" if i == n_w - 1 else ""))
    ax.axhline(-3, color="k", label="1e-3 reference")
    ax.set_ylim(-4.5, -0.5)
    ax.set_title(f"{name} | log10(update:data) per step")
    ax.set_xlabel("Step")
    ax.set_ylabel("log10(update / data)")
    ax.legend(fontsize=7, ncol=2)
fig.suptitle("Good init keeps the layers alike; gain 3 saturates them; BatchNorm undoes the bad init")
plt.tight_layout()
plt.show()

**读图**（作者的运行：种子 0；初始化时的数字 CPU 与 MPS 完全相同，训练后的数字相差不到 2 个百分点；三个种子结论相同）：

1. **好的初始化让各层"长得差不多"**。gain 5/3 时，第 2–5 层的激活分布几乎重合，饱和约 6–9%（第 1 层约 19%：它的输入是标准化后的像素，不是上一层 tanh 的输出）；各层的梯度标准差从第 5 层的 4.8e-4 到第 1 层的 7.2e-4，相差不到 2 倍。
2. **太大的初始化让 tanh 大面积饱和**。gain 3 时第 2–5 层约 40% 的输出贴在 ±1 上（第一列两端的尖峰），梯度往回传时一路被放大：第 5 层 8.7e-4，第 1 层 2.9e-3。反过来，S1 末尾提到的 PyTorch 默认初始化在这个网络里**太小**：激活标准差逐层缩到 0.08，第 1 层拿到的梯度只有第 5 层的十分之一——后面的练习里你还会用到这套诊断。
3. **BatchNorm 让前向不再受初始化尺度影响**：同样是 gain 3，加了 BN 之后饱和回到约 3%，训练 1000 步后仍是约 3%。但第四列显示 **update:data 跟着权重尺度变**：同样 lr = 0.1，BN + gain 3 的隐层约 −3.0，gain 5/3 约 −2.75。BN 抵消了权重的尺度；权重大了，同样的学习率带来的相对更新就小了。
4. **末层 W6 的 grad:data 约 1–3，update:data 前几十步在 −1 上下**。这不是 bug：我们把末层权重缩小了 10 倍，分母小；几百步后它回落到 −2 附近。
5. **−3 只是量级参照**。gain 5/3 的隐层稳定在约 −2.75，Karpathy 自己那次运行（名字数据，带 BN）的隐层在约 −2.2，都在参照线上方，训练都健康。偏离一两个数量级（比如 −5 或 −1）才值得警惕。
6. **没有 BN 时第 1 层越训越饱和**：gain 5/3 的第 1 层从初始约 19% 涨到 1000 步后约 50%（gain 3 约 77%）；它的 update:data（约 −2.2）也是隐层里最高的——lr = 0.1 对这一层偏大。BN 版各层始终约 3%。想一想：如果只把第 1 层的学习率调小，第 1 层的饱和曲线会怎样变？
7. **三种配置的验证准确率都在 0.77–0.82**，gain 3 只比 gain 5/3 低两三个百分点。只看最终准确率，你很可能觉得"差不多"，更说不出差在哪里；诊断图却在结果变坏之前就把内部的问题摆了出来。这正是体检 ④ 的价值：网络更深、训练更久时，这些问题迟早会变成结果。

四道体检的共同点：**每一道都有一个事先知道的正确答案**（2.303、≈0、0.10；体检 ④ 是饱和比例在百分之几到一成以内、各层梯度分布相近、update:data 在 10⁻³ 附近——最后这个只是量级参照，不是硬标准）。调试最怕"不知道应该看到什么"；先算出期望值，再看实际值，偏差本身就是线索。

Recipe 里还有更多（固定随机种子、先关掉数据增强、监控梯度范数与激活直方图——S1 的直方图是简化版，体检 ④ 就是它的完整版）。本周练习里，你会用这套配方排查一份**埋了 4 个雷**的训练代码：每个雷都不报错，只是结果不对。

---
## S4 · 研究方法微课：Tuning Playbook

本节依据 Google Research 的 [Deep Learning Tuning Playbook](https://github.com/google-research/tuning_playbook)（Godbole, Dahl, Gilmer, Shallue & Nado, 2023）。

### 先看一个让人不安的事实

同一个模型、同样的超参数、同样的 2000 张数据，只换随机种子（它决定初始化和每个 epoch 的打乱顺序）：

In [ ]:
seed_accs = []
for seed in range(5):
    m, _ = quick_train(X_tr, y_tr, seed=seed)
    seed_accs.append(evaluate(m, X_test, y_test, device)[0])
print("5 个种子的测试准确率:", np.round(seed_accs, 4))
print("报告格式:", experiments.format_result(seed_accs))
print(f"最好与最差差了 {100 * (max(seed_accs) - min(seed_accs)):.1f} 个百分点")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 2.8))
ax.scatter(range(5), seed_accs, s=40)
m_, s_ = experiments.mean_std(seed_accs)
ax.axhline(m_, color="gray", ls="--", label=f"Mean {m_:.3f}")
ax.axhspan(m_ - s_, m_ + s_, color="gray", alpha=0.2, label="Mean ± std")
ax.set_xticks(range(5))
ax.set_xlabel("Random seed")
ax.set_ylabel("Test accuracy")
ax.set_title("Same model, same data: the seed alone moves accuracy by points")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

任何**小于这个波动**的"改进"，都可能只是换了个种子。这就是为什么从第 6 周起要坚持"最小实验单元"：

- **至少 3 个种子**；报告**均值 ± 标准差**（`experiments.format_result`），画图带**误差棒**；
- 想给出更严格的区间时可以用 **bootstrap 置信区间**：把各种子的结果有放回地重抽上千次、每次算均值，取这些均值的 2.5% 与 97.5% 分位数作为 95% 区间——它不假设正态分布，但种子很少时区间本身也不稳，所以它不能替代多跑种子。

### 三类超参数

Playbook 把一次实验里的超参数分成三类：

| 类别 | 定义 | 本周作业里的例子 |
| --- | --- | --- |
| **科学超参数**（scientific） | 你**想测量其效应**的那个 | 网络深度 {1, 2, 4, 8} |
| **冗余超参数**（nuisance） | 本身不关心，但**必须为每个科学超参数取值单独调好**，比较才公平 | 学习率 |
| **固定超参数**（fixed） | 这一轮实验里固定不动；结论只在这些取值下成立 | 宽度 128、2 epochs、batch 64、Adam |

Playbook 举的例子恰好就是你本周的作业：研究"隐层数"的效应时，**最优学习率依赖于架构**，所以必须对每个深度分别调学习率。如果只给 2 层网络调好学习率、再把它套用到 8 层网络上，你测到的是"8 层网络 + 一个替 2 层网络选的学习率"，深度的效应和学习率失配就**混杂**在一起了。

固定超参数要写进结论的限定条件里："在 2 个 epoch 的预算下，更深的网络……"——换一个预算，结论可能不同。这不是缺陷，是诚实。

### 冗余超参数怎么调：网格搜索 vs 随机搜索

作业里你要为每个深度调学习率——最常见的冗余超参数。要调两个超参数时，最自然的做法是**网格搜索**：每个取 4 个值，两两组合跑 16 次。Bergstra & Bengio（[JMLR 2012](https://www.jmlr.org/papers/v13/bergstra12a.html)）提出了一个简单的替代：**随机搜索**——同样 16 次预算，每次从搜索范围里随机抽一组。他们的论点是：大多数问题里**只有少数几个超参数真正重要**，而哪几个重要又因数据而异。网格把预算平均分给每个维度；如果权重衰减其实不重要，16 次试验里只有 4 个不同的学习率，另外 12 次都在重复测量已知的东西。随机搜索的 16 次试验有 16 个不同的学习率。

**在对数尺度上采样**。学习率从 1e-4 到 1e-1 跨了三个数量级。在这个区间里**均匀**抽，九成样本会落在 1e-2 到 1e-1 之间。我们关心的是"差几倍"而不是"差多少"，所以要在 $\log_{10}$ 上均匀抽：先在 [−4, −1] 上均匀抽 $u$，再取 $10^u$。网格也一样：`np.logspace(-4, -1, 4)` 给出 1e-4、1e-3、1e-2、1e-1。

**实验**：本周的 784-256-10、前 2000 张、Adam 3 个 epoch（`quick_train` 的设置），在**验证集**上比较；搜索学习率 ∈ [1e-4, 1e-1] 与权重衰减 ∈ [1e-6, 1e-2]，预算都是 16 次。一次试验在 CPU 上不到 0.1 秒，所以随机搜索重复 5 次（换 5 个采样种子），顺便看看它自己的波动有多大。一次试验（cell [4] 的 `train_epochs` 已经加了 `weight_decay` 参数）：

In [ ]:
def trial(lr, wd, seed=0):
    # 一次试验：固定种子 → 784-256-10 → Adam(lr, weight_decay=wd) 训练 3 个 epoch → 验证准确率
    experiments.set_seed(seed)
    model = make_mlp().to(device)
    train_epochs(model, X_tr, y_tr, lr=lr, weight_decay=wd)
    return evaluate(model, X_val, y_val, device)[0]

写两个小工具：`log_uniform(rng, low, high, n)` 在 [low, high] 上按对数尺度均匀抽 n 个数（`rng` 是 `np.random.default_rng(...)` 生成器）；`best_so_far(vals)` 返回"迄今最好"曲线，第 k 个元素是前 k 次试验里的最大值：

In [ ]:
def log_uniform(rng, low, high, n):
    # 在这里写你的代码
    raise NotImplementedError


def best_so_far(vals):
    # 在这里写你的代码
    raise NotImplementedError


ck.log_uniform(log_uniform)
ck.best_so_far(best_so_far)

In [ ]:
BUDGET = 16
LR_RANGE, WD_RANGE = (1e-4, 1e-1), (1e-6, 1e-2)
grid = [(lr, wd) for lr in np.logspace(-4, -1, 4) for wd in np.logspace(-6, -2, 4)]   # 4 × 4，按行：先固定学习率
grid_vals = [trial(lr, wd) for lr, wd in grid]

random_runs = []                       # 5 次独立的随机搜索，每次 16 个点
for s in range(5):
    rng = np.random.default_rng(s)
    pts = list(zip(log_uniform(rng, *LR_RANGE, BUDGET), log_uniform(rng, *WD_RANGE, BUDGET)))
    random_runs.append((pts, [trial(lr, wd) for lr, wd in pts]))
random_bests = [max(vals) for _, vals in random_runs]

k = int(np.argmax(grid_vals))
print(f"网格：最好验证准确率 {grid_vals[k]:.4f}（lr = {grid[k][0]:.0e}，wd = {grid[k][1]:.0e}）；"
      f"16 次试验里只有 {len({lr for lr, _ in grid})} 个不同的学习率")
print("随机搜索 5 次，每次的最好:", np.round(random_bests, 4), "→", experiments.format_result(random_bests))
ck.search_results(grid_vals, random_bests)

In [ ]:
g, v_g = np.array(grid), np.array(grid_vals)
r0, v_r0 = np.array(random_runs[0][0]), np.array(random_runs[0][1])
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
ax = axes[0]
sc = ax.scatter(g[:, 0], g[:, 1], c=v_g, marker="s", s=80, cmap="viridis", vmin=0.5, vmax=0.82)
ax.scatter(r0[:, 0], r0[:, 1], c=v_r0, marker="o", s=80, cmap="viridis", vmin=0.5, vmax=0.82, edgecolors="k")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Learning rate")
ax.set_ylabel("Weight decay")
ax.set_title("Grid (squares) vs random search 1 (circles)")
fig.colorbar(sc, ax=ax, label="Validation accuracy")
ax = axes[1]
ax.scatter(g[:, 0], v_g, marker="s", s=60, label="Grid: 4 distinct learning rates")
ax.scatter(r0[:, 0], v_r0, marker="o", s=40, label="Random: 16 distinct learning rates")
ax.set_xscale("log")
ax.set_xlabel("Learning rate")
ax.set_ylabel("Validation accuracy")
ax.set_title("The same trials projected onto the learning rate axis")
ax.legend(fontsize=8)
ax = axes[2]
curves = np.array([best_so_far(v) for _, v in random_runs])
t = np.arange(1, BUDGET + 1)
ax.plot(t, curves.mean(axis=0), "o-", ms=3, label="Random (mean of 5 searches)")
ax.fill_between(t, curves.min(axis=0), curves.max(axis=0), alpha=0.2, label="Random (min to max)")
ax.plot(t, best_so_far(grid_vals), "s-", ms=3, label="Grid (row by row)")
ax.set_ylim(0.70, 0.82)
ax.set_xlabel("Trial")
ax.set_ylabel("Best validation accuracy so far")
ax.set_title("Best-so-far curve")
ax.legend(fontsize=8)
fig.suptitle("Random search covers 16 learning rates instead of 4; in 2-D the final best is about the same")
plt.tight_layout()
plt.show()

**诚实地读结果**（作者的运行；CPU 与 MPS 相差不到 0.1 个百分点）：

- **两个超参数时，随机搜索只赢了一点点**：网格最好 0.7975，5 次随机搜索 0.801 ± 0.004，差距比 S4 开头"只换种子"的波动（约 2 个百分点）还小。调研时把随机搜索重复了 20 次，有 15% 输给网格。Tuning Playbook 也明说："In 1-2 dimensions, grid search is also acceptable"——一两个维度时网格也可以接受，维度再多就不行了（它引的正是 Bergstra & Bengio）。
- **真正说明问题的是中间那张投影图**：网格的 16 次试验在学习率轴上叠成 4 列，随机搜索的 16 个点却描出了整条"学习率—准确率"曲线：太小学不动，1e-3 前后最好，超过 1e-2 就明显下滑。这里学习率决定一切，权重衰减几乎无关紧要，网格有 12 次试验花在了不重要的维度上。右图也一样：按行跑网格时，前 4 次都是 lr = 1e-4，第 5 次才跳上去。
- 维度一多，差别就拉开了。下一格（可选，CPU 约 7 秒）再加两个超参数：dropout ∈ [0, 0.5]、隐层宽度 ∈ [64, 512]。预算还是 16 次，4 个维度的网格每维只能取 2 个值（2⁴ = 16）。

In [ ]:
# （可选，CPU 约 7 秒）同样 16 次预算，搜索 4 个超参数：学习率、权重衰减、dropout、隐层宽度
from scipy.stats import qmc

LO = np.array([-4.0, -6.0, 0.0, np.log2(64)])          # log10(lr)、log10(wd)、dropout、log2(宽度) 的下界
HI = np.array([-1.0, -2.0, 0.5, np.log2(512)])         # 上界


def trial4(u, seed=0):
    # u ∈ [0, 1]^4 线性映射到上面的范围（对学习率、权重衰减、宽度来说就是对数尺度），再跑一次试验
    x = LO + np.asarray(u) * (HI - LO)
    lr, wd, p, width = 10 ** x[0], 10 ** x[1], float(x[2]), int(round(2 ** x[3]))
    experiments.set_seed(seed)
    model = nn.Sequential(nn.Linear(784, width), nn.ReLU(), nn.Dropout(p), nn.Linear(width, 10)).to(device)
    train_epochs(model, X_tr, y_tr, lr=lr, weight_decay=wd)
    return evaluate(model, X_val, y_val, device)[0]


corners = np.array([[a, b, c, d] for a in (0, 1) for b in (0, 1) for c in (0, 1) for d in (0, 1)], dtype=float)
search4 = {
    "2^4 grid at the range ends": [max(trial4(u) for u in corners)],
    "2^4 grid at cell centres": [max(trial4(0.25 + 0.5 * u) for u in corners)],
    "Random (3 searches)": [max(trial4(u) for u in np.random.default_rng(s).random((16, 4))) for s in range(3)],
    "Sobol quasi-random (3 searches)": [max(trial4(u) for u in qmc.Sobol(d=4, scramble=True, seed=s).random(16))
                                        for s in range(3)],
}
for name, bests in search4.items():
    print(f"{name:32s} 最好验证准确率 {np.round(bests, 4)}")

四个维度时（作者的运行；括号里是调研时各重复 10 次的结果）：网格取范围的端点（`np.logspace` 的默认写法）只有 0.752——学习率只剩 1e-4 和 1e-1 两档，恰好都不好；同样的网格挪到每个格子的中心约 0.80；随机搜索 3 次在 0.79–0.82 之间（10 次：0.805 ± 0.009，最差 0.790）。**网格的成败取决于格点恰好放在哪里，而你事先并不知道好区域在哪。**做过实验设计的话，可以把 2⁴ 网格看成 2⁴ 析因设计：每个因素只有两个水平，水平选错了就什么也看不到。

Playbook 在"探索阶段"推荐的默认做法是**准随机搜索**（quasi-random search），例如 SciPy 的 `qmc.Sobol`：它可以看成"打乱并抖动过的网格"，比纯随机铺得更均匀，不同重复之间的波动也最小（10 次：0.805 ± 0.004，最差 0.800）。

还剩最后一个坑。

In [ ]:
def retrain(lr, wd, seeds=(1, 2, 3)):
    # 用新种子重训选中的配置，返回 (验证准确率列表, 测试准确率列表)
    val, test = [], []
    for s in seeds:
        experiments.set_seed(s)
        m = make_mlp().to(device)
        train_epochs(m, X_tr, y_tr, lr=lr, weight_decay=wd)
        val.append(evaluate(m, X_val, y_val, device)[0])
        test.append(evaluate(m, X_test, y_test, device)[0])
    return val, test


searches = [("网格", (grid, grid_vals))] + [(f"随机搜索 {s + 1}", run) for s, run in enumerate(random_runs)]
drops = []
print("搜索        选中的 lr / wd       搜索时的验证 | 换 3 个新种子重训：验证          测试")
for name, (pts, vals) in searches:
    k = int(np.argmax(vals))
    val, test = retrain(*pts[k])
    drops.append(vals[k] - np.mean(val))
    print(f"{name:8s} {pts[k][0]:.1e} / {pts[k][1]:.1e}     {vals[k]:.4f}    | "
          f"{experiments.format_result(val)}   {experiments.format_result(test)}")
print(f"重训后的验证准确率比搜索时平均低 {100 * np.mean(drops):.2f} 个百分点（{len(drops)} 次搜索里 {sum(d > 0 for d in drops)} 次变低）")

**赢家诅咒**：从 16 次试验里挑最大值，这个最大值本身就是乐观的估计——挑中的那一次，可能只是那个种子运气好。上一格把 6 次搜索（网格 + 5 次随机）各自选中的配置换 3 个新种子重训：作者在 CPU 上的运行里，6 次全都变低，验证准确率平均低约 0.7 个百分点。掉得最多的是随机搜索 1：它选中的 lr ≈ 8e-3 落在学习率曲线开始下滑的地方，搜索时 0.7935，重训后验证 0.784、测试只有 0.776——当时那个种子恰好走运。（在 MPS 上，各次搜索的最好值可能有并列或换位，选中的配置会不一样，但"重训后普遍变低"的规律不变。）Playbook 有一句类似的提醒：即使做 20 次试验，特别走运和特别不走运的两次搜索之间的差别，也很可能大于同一配置换种子重训的波动。

所以：

1. 冗余超参数用随机（或准随机）搜索在**对数尺度**上调；一两个超参数时网格也可以接受；
2. 搜索只在**验证集**上进行；
3. 选出配置之后，用**新种子**重训多次，在**测试集**上报告均值 ± 标准差——不要直接报告搜索时的最好验证准确率。

作业里"每个深度在 {1e-3, 3e-3} 上选学习率、再报告 3 个种子的测试准确率"，就是这套流程的缩小版（它在测试集上报告，但没有换新种子重训：选学习率和报告用的是同一批种子。想一想这会带来多大的乐观偏差）。

### 从本周起：可以用 AI 编程助手写代码

前八周不用 AI 写代码，是为了让训练循环、数组操作、调试这些基本功长在你自己手上。从本周起，你可以用 AI 编程助手（Copilot、Claude、ChatGPT……）写代码，但请守住两条约定：

1. **每份作业附一段"代码讲解"**：用你自己的话逐段解释关键代码在做什么、为什么这样写，并注明哪些部分是 AI 生成的；
2. **自我讲解**：每份作业做完，随机挑一段代码，不看注释讲清它在做什么，再改一处、先预测结果再运行。讲不清楚的代码，视同不是你写的。

AI 让你写得更快；**判断它写得对不对**——五步顺序、eval 模式、损失函数的输入格式、种子与划分——仍然是你的工作。今天的诊断配方，就是你审查 AI 代码的检查清单。

### 本周小结

- 优化器：SGD 一个步长管所有方向；Momentum 攒惯性抵消震荡；Adam 给每个参数单独的步长——默认起点
- 学习率太小走不动、太大停在高处或发散；batch size 决定梯度噪声（∝ $1/\sqrt{B}$）和速度，不拿它调验证表现
- 初始化：随机打破对称；方差要让信号逐层保持（Xavier 配线性/tanh，Kaiming 配 ReLU）
- 过拟合的签名：验证损失先降后升；Dropout、强权重衰减、早停能收窄 gap（主要靠拉低训练准确率，测试准确率几乎不动），BatchNorm 在这个规模上没有收窄（本机 CPU 实测 gap 0.179，baseline 0.174）；最好的正则化是更多数据
- Dropout 与 BatchNorm 在 train / eval 两种模式下行为不同：BatchNorm 训练时用 batch 统计并更新 running 均值/方差，评估时用 running 统计——评估前必须 `model.eval()`
- 诊断配方四道体检：初始损失 ≈ ln(类别数)、过拟合单个 batch、输入全零的基线、打开网络看内部（饱和比例、各层梯度、grad:data、update:data 约 10⁻³ 的量级参照）；BatchNorm 抵消坏初始化，但相对更新量随权重尺度变
- 科学 / 冗余 / 固定超参数：冗余超参数要为每个科学超参数取值单独调；≥3 种子、均值±标准差、误差棒
- 冗余超参数用随机（或准随机）搜索在对数尺度上调，一两个维度时网格也可以接受；搜索时最好的那次是乐观估计，要用新种子重训、在测试集上报告
- 从本周起可用 AI 写代码，但要写"代码讲解"，并给自己讲一遍